<a href="https://colab.research.google.com/github/Yoel-Apaza/10-practica/blob/main/analisisDeDatosClaidadDeVida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- [CONFIGURACIÓN DE ARCHIVOS GENERAL] ---
# Master list of all files to process from 2018 to 2024
archivos_maestro = [
    'Enaho01B-2019-1.csv', 'Enaho01-2020-100.csv',
    'Enaho01-2021-100.csv', 'Enaho01-2022-100.csv', 'Enaho01-2023-100.csv', 'Enaho01-2024-100.csv'
]

def procesar_datos_unificado(nombre_archivo):
    """
    Procesa un archivo CSV de ENAHO, extrae el año, mapea departamentos
    y calcula todos los indicadores de calidad de vida y equipamiento.
    Retorna un DataFrame con datos a nivel de hogar.
    """
    sep = '|' if '01B' in nombre_archivo else ','

    # Corrected year extraction logic
    # Assumes filename format like 'Enaho01B-YYYY-X.csv' or 'Enaho01-YYYY-XXX.csv'
    try:
        anio = int(nombre_archivo.split('-')[1].split('.')[0]) # Ensure it handles both formats
    except Exception as e:
        print(f"ERROR al extraer año de {nombre_archivo}: {e}. Asignando 0.")
        anio = 0

    print(f"Procesando año: {anio} desde {nombre_archivo}...")

    try:
        df = pd.read_csv(nombre_archivo, sep=sep, encoding='latin-1', low_memory=False)
        df.columns = df.columns.str.upper()
        df['AÑO'] = anio
    except FileNotFoundError:
        print(f"ERROR: Archivo {nombre_archivo} no encontrado. Saltando.")
        return pd.DataFrame()
    except Exception as e:
        print(f"ERROR procesando {nombre_archivo}: {e}. Saltando.")
        return pd.DataFrame()

    # Mapeo de Departamentos
    df['UBIGEO'] = df['UBIGEO'].astype(str).str.zfill(6)
    dep_names = {
        '01':'AMAZONAS','02':'ANCASH','03':'APURIMAC','04':'AREQUIPA','05':'AYACUCHO',
        '06':'CAJAMARCA','07':'CALLAO','08':'CUSCO','09':'HUANCAVELICA','10':'HUANUCO',
        '11':'ICA','12':'JUNIN','13':'LA LIBERTAD','14':'LAMBAYEQUE','15':'LIMA',
        '16':'LORETO','17':'MADRE DE DIOS','18':'MOQUEGUA','19':'PASCO','20':'PIURA',
        '21':'PUNO','22':'SAN MARTIN','23':'TACNA','24':'TUMBES','25':'UCAYALI'
    }
    df['DEPARTAMENTO'] = df['UBIGEO'].str[:2].map(dep_names)
    # Ensure all departments are in uppercase for consistent mapping
    df['DEPARTAMENTO'] = df['DEPARTAMENTO'].str.upper()


    # --- CALCULAR INDICADORES A NIVEL DE HOGAR --- (Assign 0 if column not present, np.nan for Dormitorios)

    # P1121: Internet (1=Sí)
    if 'P1121' in df.columns:
        df['INTERNET'] = np.where(df['P1121'] == 1, 1, 0)
    else:
        df['INTERNET'] = 0 # Assume 0 if not available

    # P110: Agua por Red Pública (1=Red Pública, 2=Pilón)
    if 'P110' in df.columns:
        df['AGUA_RED'] = np.where(df['P110'].isin([1, 2]), 1, 0)
    else:
        df['AGUA_RED'] = 0 # Assume 0 if not available

    # P103: Material del Piso (6=Tierra)
    if 'P103' in df.columns:
        df['PISO_TIERRA'] = np.where(df['P103'] == 6, 1, 0)
    else:
        df['PISO_TIERRA'] = 0 # Assume 0 if not available

    # P101: Tipo de Vivienda (1=Casa independiente)
    if 'P101' in df.columns:
        df['CASA_INDIP'] = np.where(df['P101'] == 1, 1, 0)
    else:
        df['CASA_INDIP'] = 0 # Assume 0 if not available

    # P114F: Tenencia de TV (1=Sí)
    if 'P114F' in df.columns:
        df['TIENE_TV'] = np.where(df['P114F'] == 1, 1, 0)
    else:
        df['TIENE_TV'] = 0 # Assume 0 if not available

    # P114G: Tenencia de Refrigeradora (1=Sí)
    if 'P114G' in df.columns:
        df['TIENE_REFRIGERADORA'] = np.where(df['P114G'] == 1, 1, 0)
    else:
        df['TIENE_REFRIGERADORA'] = 0 # Assume 0 if not available

    # P104: Número de Habitaciones (para Hacinamiento)
    if 'P104' in df.columns:
        df['DORMITORIOS'] = pd.to_numeric(df['P104'], errors='coerce')
    else:
        df['DORMITORIOS'] = np.nan # Data not available for this year/module, keep NaN for numerical mean

    return df[['AÑO', 'DEPARTAMENTO', 'INTERNET', 'AGUA_RED', 'PISO_TIERRA',
               'CASA_INDIP', 'TIENE_TV', 'TIENE_REFRIGERADORA', 'DORMITORIOS']]

# --- [PROCESAR TODOS LOS ARCHIVOS] ---
lista_df_hogares = []
for f in archivos_maestro:
    df_temp = procesar_datos_unificado(f)
    if not df_temp.empty:
        lista_df_hogares.append(df_temp)

df_hogares_combinado = pd.concat(lista_df_hogares, ignore_index=True)

# --- [AGREGACIONES PARA VISUALIZACIONES] ---

# Agregación por Año y Departamento para indicadores de calidad de vida (similar a df_full_aggregated de tI_SjJDSyw6a)
df_calidad_region_anual = df_hogares_combinado.groupby(['AÑO', 'DEPARTAMENTO']).agg(
    INTERNET=('INTERNET', 'mean'),
    AGUA_RED=('AGUA_RED', 'mean'),
    PISO_TIERRA=('PISO_TIERRA', 'mean')
).reset_index()

for col in ['INTERNET', 'AGUA_RED', 'PISO_TIERRA']:
    df_calidad_region_anual[col] = (df_calidad_region_anual[col] * 100).round(2)


# Agregación anual de tenencia y servicios (similar a resumen_anual de rJSz0zE14_HO)
df_tenencia_servicios_anual = df_hogares_combinado.groupby('AÑO').agg(
    CASA_INDIP=('CASA_INDIP', 'mean'),
    INTERNET=('INTERNET', 'mean'),
    AGUA_RED=('AGUA_RED', 'mean'),
    TIENE_TV=('TIENE_TV', 'mean'),
    TIENE_REFRIGERADORA=('TIENE_REFRIGERADORA', 'mean')
).reset_index()

# Convertir a porcentaje (only for non-NaN values)
for col in df_tenencia_servicios_anual.columns[1:]:
    df_tenencia_servicios_anual[col] = (df_tenencia_servicios_anual[col] * 100).round(2)


# Agregación para Hacinamiento (similar a hacinamiento de rJSz0zE14_HO)
df_hacinamiento_anual = df_hogares_combinado.groupby(['AÑO', 'DEPARTAMENTO'])['DORMITORIOS'].mean().reset_index()


# --- [VISUALIZACIONES] ---
repo_geojson = 'https://raw.githubusercontent.com/reydelperu/peru-geojson/master/peru_departamental_simple.json'


# 1. Mapa de Calor: Evolución del Internet por Región (2018-2024) (de tI_SjJDSyw6a)
pivot_internet_heatmap = df_calidad_region_anual.pivot(index="DEPARTAMENTO", columns="AÑO", values="INTERNET")

fig_heatmap = px.imshow(pivot_internet_heatmap,
                labels=dict(x="Año", y="Departamento", color="% Internet"),
                x=pivot_internet_heatmap.columns,
                y=pivot_internet_heatmap.index,
                title="1. Mapa de Calor: Evolución del Internet por Región (2018-2024)",
                color_continuous_scale="Viridis",
                aspect="auto")
fig_heatmap.show()


# 2. Mapa del Perú: Acceso a Internet 2024 (de tI_SjJDSyw6a, ligeramente modificado para usar df_calidad_region_anual)
df_internet_2024 = df_calidad_region_anual[df_calidad_region_anual['AÑO'] == 2024]
fig_mapa_2024_t1 = px.choropleth(df_internet_2024,
                    geojson=repo_geojson,
                    locations='DEPARTAMENTO',
                    featureidkey='properties.NOMBDEP',
                    color='INTERNET',
                    color_continuous_scale="Reds",
                    scope="south america",
                    title="2. Mapa del Perú: Acceso a Internet 2024 (de tI_SjJDSyw6a)")

fig_mapa_2024_t1.update_geos(fitbounds="locations", visible=False)
fig_mapa_2024_t1.show()


# 3. Gráfico de Dispersión de Mejora para Pisos de Tierra (Actualizado para 2020 vs 2024)
comp_piso_scatter = df_calidad_region_anual[df_calidad_region_anual['AÑO'].isin([2020, 2024])].pivot(
    index='DEPARTAMENTO', columns='AÑO', values='PISO_TIERRA'
).reset_index()

# Ensure the columns for the scatter plot exist
if 2020 in comp_piso_scatter.columns and 2024 in comp_piso_scatter.columns:
    fig_scatter_piso = px.scatter(comp_piso_scatter, x=2020, y=2024, text='DEPARTAMENTO',
                     title="3. Comparación: % de Pisos de Tierra (2020 vs 2024)",
                     labels={2020: '% Tierra en 2020', 2024: '% Tierra en 2024'})
    max_val = max(comp_piso_scatter[2020].max(), comp_piso_scatter[2024].max())
    fig_scatter_piso.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val, line=dict(color="Red", dash="dash"))
    fig_scatter_piso.update_traces(textposition='top center')
    fig_scatter_piso.show()
else:
    print("No hay datos suficientes para comparar Pisos de Tierra entre 2020 y 2024.")


# 4. Tabla Comparativa de Tenencia y Servicios (%) (de rJSz0zE14_HO)
print("\n4. TABLA COMPARATIVA DE TENENCIA Y SERVICIOS (%)")
print(df_tenencia_servicios_anual)


# 5. Gráfico de Barras: Evolución de Servicios y Equipamiento (2020-2024) (de rJSz0zE14_HO)
df_melt_barras = df_tenencia_servicios_anual.melt(id_vars='AÑO', var_name='Indicador', value_name='Porcentaje')
fig_barras_equipamiento = px.bar(df_melt_barras, x='AÑO', y='Porcentaje', color='Indicador', barmode='group',
                   title="5. Evolución de Servicios y Equipamiento (2020-2024)")
fig_barras_equipamiento.show()


# 6. Mapa de Calor: Acceso a Internet por Región (2024) (de rJSz0zE14_HO, similar a #2, but from a different source context, let's keep it consistent)
# Using df_internet_2024 which was prepared for map #2.
fig_mapa_2024_r2 = px.choropleth(df_internet_2024,
                    geojson=repo_geojson,
                    locations='DEPARTAMENTO',
                    featureidkey='properties.NOMBDEP',
                    color='INTERNET',
                    color_continuous_scale="Viridis",
                    title="6. Mapa de Calor: Acceso a Internet por Región (2024) (de rJSz0zE14_HO)")
fig_mapa_2024_r2.update_geos(fitbounds="locations", visible=False)
fig_mapa_2024_r2.show()


# 7. Hacinamiento: Distribución del Espacio (Habitaciones) por Año (de rJSz0zE14_HO)
fig_hacinamiento_box = px.box(df_hacinamiento_anual, x="AÑO", y="DORMITORIOS", color="AÑO",
                title="7. Distribución del Espacio (Habitaciones) en Hogares por Año y Departamento")
fig_hacinamiento_box.show()


# 8. Mapa Animado de Evolución de Internet en Perú (de Y95jotLo7DSu)
# This uses df_calidad_region_anual which has 'AÑO', 'DEPARTAMENTO', 'INTERNET' (as PORCENTAJE)
fig_animated_internet_map = px.choropleth(
    df_calidad_region_anual,
    geojson=repo_geojson,
    locations='DEPARTAMENTO',
    featureidkey='properties.NOMBDEP',
    color='INTERNET', # Already a percentage from df_calidad_region_anual
    animation_frame='AÑO',
    color_continuous_scale="Viridis",
    range_color=(0, 100),
    labels={'INTERNET': '% Acceso a Internet'},
    title="8. Evolución Animada del Acceso a Internet en Perú por Departamento (2018-2024)"
)

fig_animated_internet_map.update_geos(fitbounds="locations", visible=False)
fig_animated_internet_map.update_layout(height=600)
fig_animated_internet_map.write_html("Mapa_Internet_Peru_Animado.html")

print("\n¡Proceso completado! Se han generado todas las visualizaciones y el mapa animado 'Mapa_Internet_Peru_Animado.html'.")

# --- AL FINAL DE TU CÓDIGO DE PYTHON ---
# Guardamos el dataframe combinado en un solo archivo para R
df_hogares_combinado.to_csv("enaho_consolidada_2018_2024.csv", index=False, encoding='latin-1')

print("\n¡Archivo para R generado con éxito: 'enaho_consolidada_2018_2024.csv'!")

Procesando año: 2019 desde Enaho01B-2019-1.csv...
Procesando año: 2020 desde Enaho01-2020-100.csv...
Procesando año: 2021 desde Enaho01-2021-100.csv...
Procesando año: 2022 desde Enaho01-2022-100.csv...
Procesando año: 2023 desde Enaho01-2023-100.csv...
Procesando año: 2024 desde Enaho01-2024-100.csv...



4. TABLA COMPARATIVA DE TENENCIA Y SERVICIOS (%)
    AÑO  CASA_INDIP  INTERNET  AGUA_RED  TIENE_TV  TIENE_REFRIGERADORA
0  2019         0.0       0.0       0.0       0.0                  0.0
1  2020        96.5      97.0      91.5       0.0                  0.0
2  2021        77.0      78.0      80.0       0.0                  0.0
3  2022        91.0      75.0      87.0       0.0                  0.0
4  2023        92.0      93.0      95.0       0.0                  0.0
5  2024        97.0      86.0      84.0       0.0                  0.0



¡Proceso completado! Se han generado todas las visualizaciones y el mapa animado 'Mapa_Internet_Peru_Animado.html'.

¡Archivo para R generado con éxito: 'enaho_consolidada_2018_2024.csv'!
